# Routing

While a sequential workflow, as seen in the Prompt Chaining pattern, is foundational to think about multi-step agent applications, it has several limitations:
- It lacks the ability to make decisions based on context. 
- Without a mechanism to choose the correct tool or sub-process for a specific task, the system remains rigid and non-adaptative. 
- It makes difficult to build sophisticated applications that can manage the complexity and variability of real-world user requests.

The Routing pattern provides a solution by introducing conditional logic, enabling the system to first analyze an incoming query to determine its intent or nature. Based on this analysis, the agent dynamically directs the flow of control to the most appropriate specialized tool, function, or sub-agent.



## Implementation with Flyte v2 + the Agent harness

This notebook refactors the original LangChain + LangGraph routing example into a Flyte v2 `Agent`. Instead of a hand-written LLM classifier feeding a Python `if/else`, **the routing _is_ the agent**: each route is a tool, and the harness classifies the request and dispatches to exactly one tool in a single managed step.

#### LangGraph vs Flyte v2 + Agent harness

| Aspect | LangGraph | Flyte v2 + `Agent` harness |
|--------|-----------|----------------------------|
| **Routing logic** | Explicit graph with nodes/edges | `Agent` selects a tool — one tool == one route |
| **Classifier** | A node you write and prompt | Built into the harness' tool-choice step |
| **State** | Shared typed dict flowing through nodes | Typed `AgentResult` |
| **Human-in-the-loop** | Built-in graph interrupts | `@tool(requires_approval=True)` (HITL plugin) |
| **Observability** | LangSmith | Each route handler is a nested, traced `@env.task` |
| **Execution model** | In-process only | Separate pods per task |
| **Per-task resources/images** | No | Yes |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster |

1. Install dependencies

In [ ]:
!uv pip install flyte litellm

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

2. Create a secret in Flyte for the API key (only done once):

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

3. Import dependencies and declare the resources your execution environment will need, using the `TaskEnvironment` class. This allows you to allocate precise resources and run agents on their own container with all dependencies baked in automatically:

In [ ]:
import os
import asyncio
import flyte
from flyte import TaskEnvironment, Resources, Secret
from flyte.ai.agents import Agent, AgentResult

flyte.init_from_config()

env = TaskEnvironment(
    name="routing_env",
    resources=Resources(cpu="1", memory="1Gi"),
    cache="auto",
    image=flyte.Image.from_debian_base().with_pip_packages("litellm"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
)

## Routing as tool dispatch

In the Agent harness, **routing is tool selection**. Each route becomes an `@env.task` tool with a docstring describing when to use it. The agent reads the request, picks exactly one tool, and runs it — replacing both the hand-written classifier (`router_node`) and the `if/else` dispatcher from the plain-task version.

First, define the three route handlers as tools:

Each route handler is a normal `@env.task`. Its **docstring** tells the agent when to choose it; the request text is passed straight through as the tool argument:

In [ ]:
@env.task
async def handle_booking(request: str) -> str:
    """Handle requests about booking or changing flights and hotels."""
    return f"Booking Handler processed request: '{request}'. Result: Simulated booking action."


@env.task
async def handle_info(request: str) -> str:
    """Answer general information questions (destinations, policies, schedules)."""
    return f"Info Handler processed request: '{request}'. Result: Simulated information retrieval."


@env.task
async def ask_clarification(request: str) -> str:
    """Use when the request is unclear or fits no other route; ask the user to clarify."""
    return f"Coordinator could not delegate request: '{request}'. Please clarify."

### Build the routing agent

#### From classifier + if/else to one Agent

The plain-task version needed two extra functions: `router_node` (an LLM call returning a magic string) and a `route_request` `if/else` mapping that string to a handler. The Agent harness folds both into a single declarative object — the tools _are_ the routes, and the harness performs the classify-and-dispatch step for you.

In [ ]:
router_agent = Agent(
    name="request-router",
    model="claude-haiku-4-5",
    instructions=(
        "You are a customer-service router. Classify each incoming request and call "
        "exactly ONE tool to handle it:\n"
        "- booking or changing flights/hotels -> handle_booking\n"
        "- general information questions -> handle_info\n"
        "- anything unclear or out of scope -> ask_clarification\n"
        "Pass the user's original request text to the tool, then return its result."
    ),
    tools=[handle_booking, handle_info, ask_clarification],
    max_turns=3,
)


@env.task
async def route_request(request: str) -> str:
    """Route a single request: the agent classifies it and dispatches to one handler."""
    result: AgentResult = await router_agent.run.aio(request)
    if result.error:
        raise RuntimeError(result.error)
    return result.summary

7. Run the task:

In [6]:
run = flyte.run(route_request, request="Find tickets to Amsterdam")
run.wait()
print(run.outputs()[0])
print(run.url)


> Building 1 image...

> Building image flyte for environment routing_env

i Image localhost:30000/flyte:b7e21ad4642010db89f88189de935c8b already exists, skipping build

✓ Built image for environment routing_env: localhost:30000/flyte:b7e21ad4642010db89f88189de935c8b

Output()

Booking Handler processed request: 'Find tickets to Amsterdam'. Result: Simulated booking action.
http://localhost:30080/v2/domain/development/project/flytesnacks/runs/r22nnfw6lftm72js6wl7


## Scaling the pattern

1. Make it batchable for multiple requests:

In [ ]:
@env.task
async def route_requests_batch(requests: list[str]) -> list[str]:
    """Route a batch of requests in parallel with bounded concurrency."""
    sem = asyncio.Semaphore(20)

    async def _one(req: str) -> str:
        async with sem:
            return await route_request(req)

    tasks = [_one(r) for r in requests]
    return list(await asyncio.gather(*tasks))